In [7]:
import requests
from bs4 import BeautifulSoup
import re
import csv

def scrape_eecs_faculty():
    url = "https://www2.eecs.berkeley.edu/Faculty/Lists/CS/faculty.html"
    # Headers make our script look like a regular web browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to fetch page. Status code: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 🎯 TARGETING THE EXACT CLASS FROM YOUR SCREENSHOT
    containers = soup.find_all('div', class_='cc-image-list__item__content')
    
    results = []

    for container in containers:
        # 1. Extract Name (Inside the <h3>)
        name_tag = container.find('h3')
        if not name_tag:
            continue
        name = name_tag.get_text(strip=True)
        
        # 2. Find the <p> tag holding the position and email
        p_tag = container.find('p')
        if p_tag:
            # Extract Position (The first <strong> tag inside the <p>)
            position_tag = p_tag.find('strong')
            position = position_tag.get_text(strip=True) if position_tag else "N/A"
            
            # Extract Email (Using Regex to pull it out of the raw text)
            p_text = p_tag.get_text(" ", strip=True)
            email_match = re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', p_text)
            email = email_match.group(0) if email_match else "N/A"
        else:
            position, email = "N/A", "N/A"

        results.append({
            'Name': name,
            'Position': position,
            'Email': email
        })

    return results

# Run the scraper
data = scrape_eecs_faculty()

if data:
    print(f"Successfully found {len(data)} faculty members!")
    
    # Save to CSV
    filename = 'berkeley_cs_faculty.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    print(f"Data saved to {filename}")
    
    # Print the first 3 to verify it worked
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)
else:
    print("No data found. The page might be blocking the request.")

Successfully found 134 faculty members!
Data saved to berkeley_cs_faculty.csv

--- Preview ---
{'Name': 'Pieter Abbeel', 'Position': 'Professor', 'Email': 'pabbeel@cs.berkeley.edu'}
{'Name': 'Ahmed Alaa', 'Position': 'Below The Line Assistant Professor', 'Email': 'amalaa@berkeley.edu'}
{'Name': 'Krste AsanoviÄ\x87', 'Position': 'Professor Emeritus, Professor in the Graduate School', 'Email': 'krste@berkeley.edu'}
